# Synthetic two-speaker conversations for caramel

Run from the repo root. A text teacher (Ollama or a local MLX model) writes the conversation, two Gradium Phonon voices
(on-device TTS) speak it, and the generator places both on one timeline with barge-ins, hold scenes (*wait*,
*listen*) and backchannels. caramel never sees text: the text and word timestamps only decide the timing.

**Output, in three steps:**
1. Generation → `data/synthetic/episodes/NNNNNN/` (`assistant.wav`, `user.wav`, `dialogue.wav`, `annotations.json`)
   plus the generator's own `train.jsonl` / `validation.jsonl`, which it uses to resume. Not read by caramel.
2. Export → `data/synthetic/caramel_{train,validation}.jsonl`, `prepare.py --manifest` rows: `assistant.wav` is speaker
   A, `user.wav` speaker B.
3. Tokenize → `data/synthetic_tokens_{train,validation}/` (`tokens.bin`, `index.npy`, `meta.json`), the token stores
   `train.py` reads. Every conversation is trained from both sides.

### Resume an interrupted run
Keep `OUTPUT_DIR` and the settings unchanged and rerun. Saved episodes and speech caches are reused. Timed-out
teacher requests go to `teacher_failures.jsonl`; retry them with `generator.generate(..., retry_failed=True)`.

In [ ]:
!uv pip install openai datasets ptts mlx-vlm tqdm

In [1]:
import sys
from pathlib import Path
from openai import OpenAI
from synthetic.generator import SyntheticConfig, SyntheticConversationGenerator
from synthetic.phonon_speech import PhononSpeech, PhononVoice, phonon_voices
from synthetic.teacher import ChatTeacher, MLXTeacher, load_prompt_corpus
from synthetic.caramel_export import write_caramel_manifests

In [2]:
OUTPUT_DIR = Path("data/synthetic")
MAX_SAMPLES = None # None = whole prompt corpus
SYSTEM_PROMPT = "You are JOSIE, a voice assistant. Your replies are spoken aloud: no markdown, code or math."

TEACHER = "ollama"   # "mlx": local MLX-VLM model, "ollama": OpenAI-compatible endpoint

if TEACHER == "mlx":
    teacher = MLXTeacher("mlx-community/gemma-4-e4b-it-4bit", max_tokens=512)
elif TEACHER == "ollama":
    # Fail visibly after a stalled request instead of waiting through SDK retries.
    client = OpenAI(
        base_url="http://localhost:11434/v1", api_key="ollama",
        timeout=120.0, max_retries=0,
    )
    teacher = ChatTeacher(client, assistant_model="mistral-large-4:cloud", followup_model="deepseek-v4.1-flash:cloud")
else:
    raise ValueError(f"Unknown TEACHER: {TEACHER}")

# phonon-40m-multilingual is gated (CC BY-NC 4.0): accept the terms on Hugging Face, then `hf auth login`.
# Voices: PhononVoice(name="Zoey") from `python -m ptts --list-voices`, PhononVoice(file="voice.safetensors"),
# or PhononVoice(wav="reference.wav") if the checkpoint supports cloning.
# User voice: one is picked at random per conversation from this pool. Select by name or filter, e.g.
#   phonon_voices("Zoey", "Hugh", "Maeve")  |  phonon_voices(accent="uk")  |  phonon_voices(gender="male")
ASSISTANT_VOICE = "Zoey"
USER_VOICES = phonon_voices(exclude=[ASSISTANT_VOICE])   # all 25 US + UK catalog voices except JOSIE's
speech = PhononSpeech(
    OUTPUT_DIR / "_speech_cache",
    model="hf://gradium/phonon-40m-multilingual",   # or "hf://gradium/phonon-31m-en" (Apache 2.0)
    language="en",
    user_voice=USER_VOICES,
    assistant_voice=PhononVoice(name=ASSISTANT_VOICE),
    min_alignment_match=0.8,
)
config = SyntheticConfig(
    max_turns=32, barge_in_probability=0.25, assistant_stop_ms=750, system_prompt=SYSTEM_PROMPT, seed=42,
    wait_probability=0.10,         # "can you wait a minute?" -> silent while the user is busy, then resume
    listen_probability=0.10,       # "listen until I finish my presentation" -> silent, then feedback
    max_wait_seconds=60,           # longest timed wait ("a minute")
    check_in_probability=0.5,      # "the minute is up, can I continue?" / "not yet, I'll tell you"
    backchannel_probability=0.2,   # "mhm", "right", "yeah" in the pauses of long user turns
)


In [ ]:
prompts, source = load_prompt_corpus("mlx-community/ultrafeedback-prompts-flat-rlhf")
generator = SyntheticConversationGenerator(teacher, speech, OUTPUT_DIR, config, source_metadata=source)
# Capture all thread stacks if TTS, alignment, or the teacher stops progressing.
# The log also works in kernels whose stderr does not expose a file descriptor.
import faulthandler
stall_log = OUTPUT_DIR / "generation_stall.log"
print(f"Stall diagnostics (after 120 seconds): {stall_log.resolve()}", flush=True)
with stall_log.open("w") as diagnostic_file:
    faulthandler.dump_traceback_later(120, repeat=True, file=diagnostic_file)
    try:
        episodes = generator.generate(prompts, max_samples=MAX_SAMPLES)
    finally:
        faulthandler.cancel_dump_traceback_later()
print(f"{len(episodes)} conversations, {sum(len(e['barge_ins']) for e in episodes)} barge-ins")

Stall diagnostics (after 120 seconds): /Users/Goekdeniz.Guelmez@computacenter.com/Local-Desktop/caramel/data/synthetic/generation_stall.log
Resuming data/synthetic: 32 saved episodes reused.


Synthetic conversations:   0%|          | 44/37759 [35:51<777:08:15, 74.18s/episode, episode=44, phase=user speech, turn=1/14]      

Episode 43 skipped: Teacher 'mistral-large-4:cloud' timed out after 120 seconds


Synthetic conversations:   0%|          | 82/37759 [4:58:50<3761:00:07, 359.36s/episode, episode=82, phase=user speech, turn=1/9]       

Episode 81 skipped: Teacher 'mistral-large-4:cloud' timed out after 120 seconds


Synthetic conversations:   0%|          | 85/37759 [5:13:57<3382:55:10, 323.26s/episode, episode=85, phase=user speech, turn=1/6]       

... (4 outputs left out)

Episode 96 skipped: Teacher 'mistral-large-4:cloud' timed out after 120 seconds


Synthetic conversations:   0%|          | 98/37759 [6:45:46<3557:36:39, 340.07s/episode, episode=98, phase=user speech, turn=1/2]      

Episode 97 skipped: Teacher 'mistral-large-4:cloud' timed out after 120 seconds


Synthetic conversations:   0%|          | 99/37759 [6:57:07<2712:46:12, 259.32s/episode, episode=99, phase=assistant speech, turn=14/19]

In [ ]:
# Low-disk alternative to the cell above + the manifest/tokenize cells below: generate CHUNK_SIZE conversations, tokenize
# them, upload one parquet shard per split to HF_REPO, delete the local audio and speech cache, repeat. Resumable.
# Then train straight from the Hub (streamed, nothing stored):  python train.py --data hf://<repo> --dev hf://<repo> ...
from synthetic.hub import generate_to_hub
HF_REPO = "your-username/caramel-synthetic"   # `hf auth login` first
CHUNK_SIZE = 100
generator = SyntheticConversationGenerator(teacher, speech, OUTPUT_DIR, config, source_metadata=source)
meta = generate_to_hub(generator, prompts, HF_REPO, chunk_size=CHUNK_SIZE, max_samples=MAX_SAMPLES,
                       speech_cache=OUTPUT_DIR / "_speech_cache")
print(meta["frames"])

## Export and tokenize for caramel
Only complete episodes of at least 10 s are exported. `prepare.py` appends to an existing store, so the token folders
are rebuilt from scratch here: rerunning after more generation never duplicates conversations.

In [3]:
counts = write_caramel_manifests(OUTPUT_DIR, min_seconds=10.0)
print(counts)

{'train': 49, 'validation': 6}


In [ ]:
import shutil, subprocess
TOKENS = {split: OUTPUT_DIR.parent / f"synthetic_tokens_{split}" for split in ("train", "validation")}
for split, out in TOKENS.items():
    shutil.rmtree(out, ignore_errors=True)
    if counts[split]:
        subprocess.run([sys.executable, "prepare.py", str(OUTPUT_DIR / f"caramel_{split}.jsonl"),
                        "--manifest", "--out", str(out)], check=True)

## Train
Fine-tune `caramel-1` on the new stores (see TRAINING_M4_PRO.md for memory settings).

In [ ]:
print(f"python train.py --init ./caramel-1 --data {TOKENS['train']} --dev {TOKENS['validation']} "
      "--fpb 1500 --accum 32 --epochs 3 --lr 3e-5 --warmup 10 --eval_every 25 --save_best --out runs/synthetic")